# Notebook 01 — Data Pipeline (2025 BDB)

**Dataset:** 2025 NFL Big Data Bowl — 2022 season, weeks 1–9  
**Goal:** Load, filter, standardize, and clip all tracking + reference data into clean outputs for downstream notebooks.

## Section 1 — Imports & Paths

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# print("Project root:", PROJECT_ROOT)

## Section 2 — Load Tracking Data (Weeks 1–9)

In [27]:
# Need dropback gameId/playId pairs first — load plays now just for the filter key
plays_raw = pd.read_csv(DATA_DIR / 'plays.csv')
# filter to dropback plays (passing plays) and keep only gameId and playId for merging
dropback_plays = plays_raw.loc[plays_raw['isDropback'] == True, ['gameId', 'playId']]

# Store all filtered tracking data here
weeks = []
# Loop through tracking data for weeks 1-9, filter to dropback only plays, and then add it to a single dataframe
for w in range(1, 10):
    df = pd.read_csv(DATA_DIR / f'tracking_week_{w}.csv')
    df = df.merge(dropback_plays, on=['gameId', 'playId'], how='inner')
    weeks.append(df)
    print(f'Week {w}: {len(df):,} rows after filter')

tracking = pd.concat(weeks, ignore_index=True)
print(f'\nTotal tracking rows: {len(tracking):,}')
# print('Columns:', tracking.columns.tolist())

Week 1: 4,523,594 rows after filter
Week 2: 4,274,366 rows after filter
Week 3: 4,574,723 rows after filter
Week 4: 4,045,976 rows after filter
Week 5: 4,388,630 rows after filter
Week 6: 3,945,305 rows after filter
Week 7: 3,666,292 rows after filter
Week 8: 4,065,549 rows after filter
Week 9: 3,520,702 rows after filter

Total tracking rows: 37,005,137


## Section 3 — Load Reference Tables

In [ ]:
# Load the data files for reference
games = pd.read_csv(DATA_DIR / 'games.csv')
plays = plays_raw 
players = pd.read_csv(DATA_DIR / 'players.csv')
player_play = pd.read_csv(DATA_DIR / 'player_play.csv')

print(games.shape, plays.shape, players.shape, player_play.shape)

(136, 9) (16124, 50) (1697, 7) (354727, 50)


## Section 4 — Filter to Dropback Plays

In [ ]:
# filter out the plays to only include dropbacks (passing situations), since we are focused on a pass rushing metric for edge rushers
dropback_plays = plays.loc[plays['isDropback'] == True, ['gameId', 'playId']]

print(f'Dropback plays: {len(dropback_plays):,} of {len(plays):,} total ({len(dropback_plays)/len(plays):.1%})')


Dropback plays: 9,736 of 16,124 total (60.4%)


## Section 5 — Filter to EDGE Pass Rushers

In [ ]:
# Only include players that are initial pass rushers on dropback plays
rushers = player_play.loc[player_play['wasInitialPassRusher'] == 1, ]

# Filter to only include edge rushers (DE and OLB positions) and keep only the columns needed for merging and analysis
edge_players = players.loc[players['position'].isin(['DE', 'OLB']), ['nflId', 'displayName', 'position']]

# Merge the edge rushers with the player_play data 
# Creates a dataframe that only has edge rushers on dropback plays
edge_player_play = rushers.merge(edge_players, on='nflId', how='inner') \
                          .merge(dropback_plays, on=['gameId', 'playId'], how='inner')

print(edge_player_play['position'].value_counts())
print(f'Unique EDGE rushers: {edge_player_play["nflId"].nunique()}')

DE     13471
OLB     9874
Name: position, dtype: int64
Unique EDGE rushers: 253


## Section 6 — Coordinate Standardization

In [ ]:
# Merge the tracking data with the dropback plays to only include tracking data for dropback plays
tracking_dropbacks = tracking.merge(dropback_plays, on = ['gameId', 'playId'], how = 'inner')

# Standardize the x and direction coordinates so that all plays are going in the same direction (to the right)
tracking_dropbacks['x_std'] = np.where(tracking_dropbacks['playDirection'] == 'left', 120 - tracking_dropbacks['x'], tracking_dropbacks['x'])

# Flip the direction by 180 degrees for plays going to the left, so that all plays have direction relative to moving to the right
tracking_dropbacks['dir_std'] = np.where(tracking_dropbacks['playDirection'] == 'left', (tracking_dropbacks['dir'] + 180) % 360, tracking_dropbacks['dir'])

print(tracking_dropbacks.shape)

(37005137, 20)


## Section 7 — Snap Frame Lookup & Clip Tracking

In [ ]:
# Find the frame where the ball is snapped for unique play
snap_frames = tracking_dropbacks.loc[tracking_dropbacks['event'] == 'ball_snap'] \
    .groupby(['gameId', 'playId'])['frameId'] \
    .min().reset_index().rename(columns={'frameId': 'snapFrameId'})

# Merge the snap frame information back to the tracking data
tracking_dropbacks = tracking_dropbacks.merge(snap_frames, on=['gameId', 'playId'], how='left')

# calculate the frame offset from the snap for each row
tracking_dropbacks['frameOffset'] = tracking_dropbacks['frameId'] - tracking_dropbacks['snapFrameId']

# Only keep the frames from 0 to 25 after the snap to focus on the initial 2.5 seconds of the play to gauge the impact of the pass rusher 
# PFF uses a 2.5 second window to determine pass rush win rate
tracking_clipped = tracking_dropbacks.loc[tracking_dropbacks['frameOffset'].between(0, 25)]

print(tracking_clipped.shape)
print(f"Unique plays: {tracking_clipped.groupby(['gameId', 'playId']).ngroups:,}")

(5808236, 22)
Unique plays: 9,727


## Section 8 — Save Outputs & Verification

In [26]:
tracking_clipped.to_parquet( OUT_DIR /"tracking_clipped.parquet", index=False)
edge_player_play.to_csv(OUT_DIR / "edge_player_play.csv", index=False)

print(tracking_clipped.shape)
print(f"Unique plays: {tracking_clipped.groupby(['gameId', 'playId']).ngroups:,}")
print(f"Unique EDGE rushers: {edge_player_play['nflId'].nunique():,}")
# print(f"Output files: {list(OUT_DIR.iterdir())}")

(5808236, 22)
Unique plays: 9,727
Unique EDGE rushers: 253
